# 02_export

Export DeclipNet to ONNX and Core ML for deployment.

1. Load the unpruned model (pruning was not viable — see `02_pruning`)
2. `torch.onnx.export` with dynamic batch dim; verify numerics match PyTorch via `onnxruntime`
3. `coremltools.convert` targeting `mlprogram` / `compute_units=ALL`; verify predictions within tolerance
4. Save `.onnx` and `.mlpackage` to `models/`

In [1]:
import sys
import torch
import numpy as np
import onnx
import onnxruntime as ort
import coremltools as ct
from pathlib import Path

sys.path.insert(0, "..")
sys.path.insert(0, "../01_dnn")

from config import *
from util import DeclipNet

DEVICE = torch.device("cpu")  # export on CPU for max compatibility

H = 8
N_ATTN = 3
NUM_HEADS = 4
FFN_DIM = 256

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

scikit-learn version 1.7.2 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
Torch version 2.11.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.7.0 is the most recent version that has been tested.


In [2]:
# Load model
model = DeclipNet(H=H, N=N_ATTN, num_heads=NUM_HEADS, ffn_dim=FFN_DIM)
state_dict = torch.load(FINAL_OUT / "weighted_l1_dwt" / "best_model.pt", weights_only=True, map_location="cpu")
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"DeclipNet loaded — {n_params:,} parameters")

DeclipNet loaded — 314,001 parameters


In [3]:
# ONNX Export, verification

onnx_path = MODELS_DIR / "declipnet.onnx"

dummy = torch.randn(1, 1, BS)

torch.onnx.export(
    model,
    dummy,
    onnx_path,
    input_names=["clipped"],
    output_names=["restored"],
    dynamic_axes={"clipped": {0: "batch"}, "restored": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)
print(f"Exported ONNX model to {onnx_path}")

# Validate graph structure
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print(f"ONNX checker passed — graph has {len(onnx_model.graph.node)} nodes")

# Numeric verification against PyTorch
sess = ort.InferenceSession(str(onnx_path))

test_input = torch.randn(4, 1, BS)
pt_out = model(test_input).detach().numpy()
ort_out = sess.run(None, {"clipped": test_input.numpy()})[0]

max_diff = np.abs(pt_out - ort_out).max()
mean_diff = np.abs(pt_out - ort_out).mean()
print(f"\nONNX vs PyTorch — max diff: {max_diff:.2e}, mean diff: {mean_diff:.2e}")
assert max_diff < 1e-5, f"ONNX numeric mismatch too large: {max_diff}"
print("Numeric verification passed")

/var/folders/bq/csqgvczj6jndm459wzs6n4kw0000gn/T/ipykernel_82647/1518750191.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/Users/lane/code/ind/speech-declipping/venv/lib/python3.10/site-packages/torch/onnx/_internal/torchscript_exporter/jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/jit/passes/onnx/constant_fold.cpp:180.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/Users/lane/code/ind/speech-declipping/venv/lib/python3

Exported ONNX model to models/declipnet.onnx
ONNX checker passed — graph has 452 nodes

ONNX vs PyTorch — max diff: 5.72e-06, mean diff: 1.67e-07
Numeric verification passed


In [6]:
# Core ML conversion and verification

mlmodel_path = MODELS_DIR / "declipnet.mlpackage"

traced = torch.jit.trace(model, dummy, check_trace=False)

mlmodel = ct.convert(
    traced,
    inputs=[ct.TensorType(name="clipped", shape=(1, 1, BS))],
    outputs=[ct.TensorType(name="restored")],
    convert_to="mlprogram",
    compute_units=ct.ComputeUnit.ALL,
    minimum_deployment_target=ct.target.macOS13,
)
mlmodel.save(str(mlmodel_path))
print(f"Saved Core ML model to {mlmodel_path}")

# Numeric verification against PyTorch
test_input = torch.randn(1, 1, BS)
pt_out = model(test_input).detach().numpy()

cml_pred = mlmodel.predict({"clipped": test_input.numpy()})["restored"]

max_diff = np.abs(pt_out - cml_pred).max()
mean_diff = np.abs(pt_out - cml_pred).mean()
print(f"\nCore ML vs PyTorch — max diff: {max_diff:.2e}, mean diff: {mean_diff:.2e}")
assert max_diff < 5e-2, f"Core ML numeric mismatch too large: {max_diff}"
print("Numeric verification passed")

Converting PyTorch Frontend ==> MIL Ops: 100%|▉| 442/443 [00:00<00:00, 8077.1
Running MIL frontend_pytorch pipeline: 100%|█| 5/5 [00:00<00:00, 283.46 passe
Running MIL default pipeline: 100%|████| 95/95 [00:00<00:00, 189.11 passes/s]
Running MIL backend_mlprogram pipeline: 100%|█| 12/12 [00:00<00:00, 341.11 pa


Saved Core ML model to models/declipnet.mlpackage

Core ML vs PyTorch — max diff: 2.20e-02, mean diff: 3.08e-03
Numeric verification passed


## Summary

- **Model:** DeclipNet (unpruned), 314,001 parameters, input shape `(batch, 1, 1024)`
- **ONNX** (`models/declipnet.onnx`)
  - 452-node graph, opset 17, dynamic batch dim
  - vs PyTorch: max diff 5.72e-06, mean diff 1.67e-07 — effectively exact (float32)
- **Core ML** (`models/declipnet.mlpackage`)
  - `mlprogram` format, `compute_units=ALL`, macOS 13+ / iOS 16+
  - vs PyTorch: max diff 2.20e-02, mean diff 3.08e-03 — expected from float16 ANE pipeline
- **Notes**
  - `dynamo=False` required for ONNX export (Torch 2.11 default exporter needs `onnxscript`)
  - `check_trace=False` required for `torch.jit.trace` (MHA's SDPA backend dispatch is non-deterministic across invocations)
  - Pruning not applied — structured channel pruning collapsed quality at all sparsity levels (see `02_pruning`)